In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\ANISH\AppData\Local\Temp\ipykernel_13344\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\ANISH\OneDrive\Desktop\Projs\inteli_docs_rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
### Read all the pdfs inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")


Found 2 PDF files to process

Processing: base research paper.pdf
  ✓ Loaded 17 pages

Processing: main_paper.pdf
  ✓ Loaded 5 pages

Total documents loaded: 22


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.26; modified using iText® Core 9.0.0 (AGPL version) ©2000-2024 Apryse Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-03-07T21:21:58+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-03T23:09:32+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\base research paper.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'base research paper.pdf', 'file_type': 'pdf'}, page_content='This paper is included in the \nProceedings of the 22nd USENIX Symposium on \nNetworked Systems Design and Implementation.\nApril 28–30, 2025 • Philadelphia, PA, USA\n978-1-9391 33-46-5\nOpen access to the Proceedings of the \n22nd USENIX Symposium on Networked \nSystems Design and Implementation \nis sponsored by\nGREEN: Carbon-efficient Resource Scheduling  \nfor Machine Learning Clus

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3214.58it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\ANISH\AppData\Local\Temp\ipykernel_13344\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [6]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 131


In [7]:
# Create text chunks from the loaded PDF documents
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(all_pdf_documents)
print(f"Created {len(chunks)} chunks from {len(all_pdf_documents)} documents")

Created 131 chunks from 22 documents


In [8]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.26; modified using iText® Core 9.0.0 (AGPL version) ©2000-2024 Apryse Group NV', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-03-07T21:21:58+00:00', 'author': '', 'keywords': '', 'moddate': '2025-04-03T23:09:32+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\base research paper.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1', 'source_file': 'base research paper.pdf', 'file_type': 'pdf'}, page_content='This paper is included in the \nProceedings of the 22nd USENIX Symposium on \nNetworked Systems Design and Implementation.\nApril 28–30, 2025 • Philadelphia, PA, USA\n978-1-9391 33-46-5\nOpen access to the Proceedings of the \n22nd USENIX Symposium on Networked \nSystems Design and Implementation \nis sponsored by\nGREEN: Carbon-efficient Resource Scheduling  \nfor Machine Learning Clus

In [9]:
texts = [doc.page_content for doc in chunks]

texts

['This paper is included in the \nProceedings of the 22nd USENIX Symposium on \nNetworked Systems Design and Implementation.\nApril 28–30, 2025 • Philadelphia, PA, USA\n978-1-9391 33-46-5\nOpen access to the Proceedings of the \n22nd USENIX Symposium on Networked \nSystems Design and Implementation \nis sponsored by\nGREEN: Carbon-efficient Resource Scheduling  \nfor Machine Learning Clusters\nKaiqiang Xu and Decang Sun, iSING Lab, Hong Kong University of Science and \nTechnology; Han Tian, USTC; Junxue Zhang and Kai Chen, iSING Lab, Hong Kong \nUniversity of Science and Technology\nhttps://www.usenix.org/conference/nsdi25/presentation/xu-kaiqiang',
 'GREEN: Carbon-efficient Resource Scheduling for Machine Learning Clusters\nKaiqiang Xu1, Decang Sun 1, Han Tian 2, Junxue Zhang 1, Kai Chen 1\n1iSING Lab, Hong Kong University of Science and Technology 2USTC\nAbstract\nThis paper explores the problem of scheduling machine Learn-\ning (ML) jobs while also taking into account the reduction\

In [10]:
# Generate embeddings for the chunk texts and add to vector store
embeddings = embedding_manager.generate_embeddings(texts)
print(f"Embeddings shape: {embeddings.shape}")
# Storing in vector DB
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 131 texts...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:09<00:00,  1.92s/it]


Generated embeddings with shape: (131, 384)
Embeddings shape: (131, 384)
Adding 131 documents to vector store...
Successfully added 131 documents to vector store
Total documents in collection: 262


RAG Retrivier

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

Integrating LLM

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set")

# Initialize Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3, api_key=GEMINI_API_KEY)


In [21]:
def ask_gemini(query, retriever, llm):
    # Retrieve relevant documents (using your existing retriever)
    relevant_docs = retriever.retrieve(query, top_k=3)
    
    # Combine content from retrieved documents
    context_text = "\n\n".join([doc['content'] for doc in relevant_docs])
    
    # Construct prompt
    prompt = f"""
    You are a helpful assistant. Use the following context to answer the question.
    If the answer is not in the context, say you don't have enough information.
    
    Context:
    {context_text}
    
    Question:
    {query}
    
    Answer:
    """
    
    # Generate response
    response = llm.invoke(prompt)
    return response.content

# Example Usage:
question = "What is Carbon-aware Resource Scheduling ?"
answer = ask_gemini(question, rag_retriever, llm)
print(answer)

Retrieving documents for query: 'What is Carbon-aware Resource Scheduling ?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.63it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Carbon-aware Resource Scheduling involves shifting flexible workloads toward lower-carbon electricity in time or space. This can include geographic load balancing, temporal shifting, and carbon-intensity-aware resource management. It aims to minimize the cluster-wide carbon footprint by dynamically adjusting job priorities for peak load shifting, and by shifting high-power-consuming jobs to greener hours with lower carbon intensity.
